In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import pulp

In [29]:
class Scenario:
    # Storage
    S_capacity: float = 200 # MWh_th
    S_initial_soc: float = 200 # MWh_th
    S_max_charge: float = 15 # MW_th
    S_max_discharge: float = 15 # MW_th
    S_losses: float = 0.000125 # MWh_th per 0.25h

    # Heat Pump
    HP_max_power: float = 8 # MW_el
    HP_min_power: float = 1 # MW_el
    HP_cop: float = 3.5 # MW_th per MW_el

    # Condensing Boiler
    CB_max_power: float = 12 # MW_th
    CB_min_power: float = 2 # MW_th
    CB_efficiency: float = 0.97 # MW_th per MW_Hs
    CB_min_run_time: int = 1 # hours
    CB_min_down_time: int = 1 # hours

    # CHP
    CHP_el_max_power: float = 6 # MW_el
    CHP_el_min_power: float = 2 # MW_el
    CHP_th_max_power: float = 7 # MW_th
    CHP_el_efficiency: float = 0.4 # MW_el per MW_Hs
    CHP_th_efficiency: float = 0.48 # MW_th per MW_Hs
    CHP_min_run_time: int = 2 # hours
    CHP_min_down_time: int = 2 # hours
    CHP_startup_cost: float = 600 # EUR

    # Gas Price
    gas_price: float = 35 # EUR per MWh_Hs
    gas_co2_emission_factor: float = 0.201 # tCO2 per MWh_Hs
    gas_co2_emission_price: float = 60 # EUR per tCO2

    

heat_demand = pd.read_csv('Data\RawData_MeasuredHeadDemand.csv', index_col=0)
heat_demand.index = pd.to_datetime(heat_demand.index, utc=True).tz_convert('Europe/Berlin')
heat_demand['Measured Heat Demand[MW]'] = heat_demand['Measured Heat Demand[W]'] / 1000000
heat_demand = heat_demand[['Measured Heat Demand[MW]']].rename(columns={'Measured Heat Demand[MW]': 'Heat_Demand'})

el_price = pd.read_csv('Data\Gro_handelspreise_202403010000_202603020000_Stunde.csv', index_col=0, sep=';', decimal=',', encoding='utf-8')
el_price.index = pd.to_datetime(el_price.index, format='%d.%m.%Y %H:%M').tz_localize('Europe/Berlin', ambiguous='NaT')
el_price = el_price[['Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen']]
el_price = el_price.rename(columns={'Datum von': 'Time', 'Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen': 'El_Price'})

df = heat_demand.join(el_price, how='inner')
display(df.head())

<>:38: SyntaxWarning: invalid escape sequence '\R'
<>:43: SyntaxWarning: invalid escape sequence '\G'
<>:38: SyntaxWarning: invalid escape sequence '\R'
<>:43: SyntaxWarning: invalid escape sequence '\G'
C:\Users\Nikla\AppData\Local\Temp\ipykernel_23712\610281050.py:38: SyntaxWarning: invalid escape sequence '\R'
  heat_demand = pd.read_csv('Data\RawData_MeasuredHeadDemand.csv', index_col=0)
C:\Users\Nikla\AppData\Local\Temp\ipykernel_23712\610281050.py:43: SyntaxWarning: invalid escape sequence '\G'
  el_price = pd.read_csv('Data\Gro_handelspreise_202403010000_202603020000_Stunde.csv', index_col=0, sep=';', decimal=',', encoding='utf-8')


,Heat_Demand,El_Price
Time Point,,
2024-03-01 00:00:00+01:00,5.888522,62.04
2024-03-01 01:00:00+01:00,5.916365,61.42
2024-03-01 02:00:00+01:00,5.981230,58.14
2024-03-01 03:00:00+01:00,6.101700,57.83
2024-03-01 04:00:00+01:00,6.714678,58.30


In [ ]:
def build_MILP(s: Scenario, heat_demand: pd.Series, el_price: pd.Series):
    model = pulp.LpProblem("Heat_System_Optimization", pulp.LpMinimize)

    # Continuous Decision Variables
    HP_power = pulp.LpVariable.dicts("HP_Power", range(24), lowBound=s.HP_min_power, upBound=s.HP_max_power, cat='Continuous')
    CB_power = pulp.LpVariable.dicts("CB_Power", range(24), lowBound=s.CB_min_power, upBound=s.CB_max_power, cat='Continuous')
    CHP_el_power = pulp.LpVariable.dicts("CHP_El_Power", range(24), lowBound=0, upBound=s.CHP_el_max_power, cat='Continuous')
    CHP_th_power = pulp.LpVariable.dicts("CHP_Th_Power", range(24), lowBound=0, upBound=s.CHP_th_max_power, cat='Continuous')
    Storage_charge = pulp.LpVariable.dicts("Storage_Charge", range(24), lowBound=0, upBound=s.S_max_charge, cat='Continuous')
    Storage_discharge = pulp.LpVariable.dicts("Storage_Discharge", range(24), lowBound=0, upBound=s.S_max_discharge, cat='Continuous')
    Storage_soc = pulp.LpVariable.dicts("Storage_SoC", range(25), lowBound=0, upBound=s.S_capacity, cat='Continuous')

    # Binary Decision Variables
    CHP_on = pulp.LpVariable.dicts("CHP_On", range(24), cat='Binary')
    CB_on = pulp.LpVariable.dicts("CB_On", range(24), cat='Binary')
    HP_on = pulp.LpVariable.dicts("HP_On", range(24), cat='Binary')

    # Objective Function
    HP_cost = pulp.lpSum([HP_power[t] * el_price[t] / s.HP_cop for t in range(24)])
    CHP_cost = pulp.lpSum([CHP_el_power[t] * el_price[t] / s.CHP_el_efficiency + CHP_th_power[t] * s.gas_price / s.CHP_th_efficiency + (CHP_el_power[t] > 0) * s.CHP_startup_cost for t in range(24)])
    CB_cost = pulp.lpSum([CB_power[t] * el_price[t] / s.CB_efficiency for t in range(24)])
    
    total_cost = (pulp.lpSum([HP_on[t] * HP_cost for t in range(24)]) 
                  + pulp.lpSum([CHP_on[t] * CHP_cost for t in range(24)]) 
                  + pulp.lpSum([CB_on[t] * CB_cost for t in range(24)]))
